<a href="https://colab.research.google.com/github/IvanSSantana/ai-academic-search/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETUP DO OLLAMA (IA LOCAL)

In [ ]:
import subprocess

subprocess.run(["apt-get", "install", "zstd"])

CompletedProcess(args=['apt-get', 'install', 'zstd'], returncode=0)

Abra o terminal do Colab e cole o comando abaixo.

In [ ]:
curl -fsSL https://ollama.com/install.sh | sh

Após a instalação do Ollama acima, digite no terminal 'ollama serve' e rode os dois comandos abaixo, entretanto espere a instalação com 'ollama pull' antes de rodar 'ollama run', para isto observe no terminal os packages sendo instalados

In [ ]:
import os
import subprocess

def start_ollama_server():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "pull", "llama3.1"])

start_ollama_server()

In [ ]:
import os
import subprocess

def start_ollama_server():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "run", "llama3.1"])

start_ollama_server()

# APP

In [ ]:
!pip install agno ollama arxiv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.9 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=8b1c9f6dc4b3940c1c75c951c0e4ed5cb3f837fd91706c3cfd097ef8aa11ccfd
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [ ]:
import json
import time
import arxiv
from agno.models.ollama import Ollama
from agno.agent import Agent
from agno.tools import tool

def extrair_palavras_chave(prompt: str) -> str:
    """
    Usa o modelo LLM local (Ollama) para gerar até 5 palavras-chave em inglês
    a partir do prompt, otimizadas para busca acadêmica.
    """
    agent = Agent(
        model = Ollama(id="llama3.1"),
        instructions="Dado o seguinte tópico de pesquisa, forneça ATÉ 5 palavras-chave em inglês, podendo ser menos "
        "separadas por espaços, que melhor representem o conteúdo para busca acadêmica no arXiv. "
        "Responda apenas as palavras-chave, sem nenhum texto adicional.\n\n"
    )
    try:
        palavras = agent.run(prompt)
        # Garantir que são apenas as palavras-chave (caso o modelo adicione algo)
        # Pode-se pegar as primeiras 3 palavras
        print(f"INFO: Palavras-chave extraídas: {palavras.content}")
        return palavras.content
    except Exception as e:
        print(f"WARN: Erro ao extrair palavras-chave: {e}")
        # Fallback: usar o prompt original (primeiras 3 palavras significativas)
        return prompt


def buscar_analise_arxiv(prompt: str, max_results: int = 5) -> str:
    """
    Pesquisa artigos no arXiv usando palavras-chave extraídas do prompt.

    Args:
        prompt: Tópico ou pergunta de pesquisa.
        max_results: Quantidade de artigos a retornar (padrão 5).

    Returns:
        String formatada com títulos, autores, resumos e links dos artigos encontrados.
    """
    try:
        # Extrai palavras-chave do prompt
        query = extrair_palavras_chave(prompt)
        print(f"INFO: Pesquisando no arXiv com query: {query}")

        client = arxiv.Client(page_size=max_results, delay_seconds=3, num_retries=3)
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )

        resultados = list(client.results(search))
        if not resultados:
            print("WARN: Nenhum artigo encontrado no arXiv para a query:", query)
            return "Nenhum artigo encontrado no arXiv para este tópico."

        resposta = []
        for i, paper in enumerate(resultados, 1):
            resposta.append(f"### {i}. {paper.title}")
            resposta.append(f"**Autores:** {', '.join(a.name for a in paper.authors[:5])}")
            resposta.append(f"**Publicado em:** {paper.published.strftime('%d/%m/%Y')}")
            resposta.append(f"**Resumo:** {paper.summary}")
            resposta.append(f"**Link:** {paper.entry_id}")
            resposta.append("")

        resposta_completa = "\n".join(resposta)

        print("INFO: Artigos buscados da API ->\n", resposta_completa)
        return resposta_completa

    except Exception as e:
        print(f"WARN: Erro ao consultar o arXiv: {str(e)}")
        return f"Erro ao consultar o arXiv: {str(e)}"


# Ferramenta para o agente Agno
@tool(name="pesquisa_arxiv")
def ferramenta_pesquisa_arxiv(pergunta: str) -> str:
    """
    Ferramenta para pesquisar artigos acadêmicos no arXiv.
    Recebe uma pergunta ou tópico e retorna os artigos mais relevantes.
    """
    print(f"INFO: Ferramenta acionada com pergunta: {pergunta}")
    return buscar_analise_arxiv(pergunta)


# Criação do agente Agno
agente = Agent(
    model=Ollama(id="llama3.1"),
    tools=[ferramenta_pesquisa_arxiv],
    instructions=(
        "Use a ferramenta 'pesquisa_arxiv' para buscar informações acadêmicas. "
        "Responda sempre em português. Cite os artigos encontrados com título e link."
    ),
    markdown=True
)

if __name__ == "__main__":
    pergunta = input("Digite uma pergunta: ")
    resposta = agente.run(pergunta)
    print(resposta.content)

Digite uma pergunta: Relacione o uso de esteroides com a queda de desempenho de atletas profissionais e explique como isso afeta o cérebro. Utilize de linguagem acessível a leigos.
INFO: Ferramenta acionada com pergunta: Uso de esteroides e queda de desempenho em atletas profissionais: efeitos no cérebro
INFO: Palavras-chave extraídas: Epidemic steroid use brain damage
INFO: Pesquisando no arXiv com query: Epidemic steroid use brain damage
INFO: Artigos buscados da API ->
 ### 1. Precise measurement of CMB polarisation from Dome-C: the BRAIN and CLOVER experiments
**Autores:** M. Piat, C. Rosset, the BRAIN, CLOVER Collaboration
**Publicado em:** 22/12/2004
**Resumo:** The characterisation of CMB polarisation is one of the next challenge in observationnal cosmology. This is especially true for the so-called B-modes that are at least 3 order of magnitude lower than CMB temperature fluctuations. A precise measurement of the angular power spectrum of these B-modes will give important const